<a href="https://colab.research.google.com/github/NeilKapoor2/Predicting-Soccer-Player-Salaries/blob/main/Predicting_Player_Salaries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import pandas as pd
import os

# Download the dataset
path = kagglehub.dataset_download("crawford/us-major-league-soccer-salaries")

# Read every CSV file into one DataFrame
dfs = []

for file in os.listdir(path):
    if file.endswith(".csv"):
        season = pd.read_csv(os.path.join(path, file))

        # Store the season as a feature
        season["year"] = file.replace(".csv", "")

        dfs.append(season)

# Combine all seasons
df = pd.concat(dfs, ignore_index=True)

# Count rows before removing missing values
rows_before = len(df)

# Remove rows with missing values
df = df.dropna()

# Count rows after removing missing values
rows_after = len(df)
rows_dropped = rows_before - rows_after

# Keep only the columns needed for your model
df = df[[
    "club",
    "first_name",
    "last_name",
    "position",
    "guaranteed_compensation",
    "base_salary",
    "year"
]]

print(df.head())
print(df.shape)

print(f"Rows before dropna(): {rows_before}")
print(f"Rows after dropna(): {rows_after}")
print(f"Rows dropped: {rows_dropped}")

100%|██████████| 83.0k/83.0k [00:00<00:00, 47.5MB/s]

Extracting files...
  club first_name last_name position  guaranteed_compensation  base_salary  \
0  ATL     Miguel   Almiron        M               2297000.00    1912500.0   
1  ATL      Mikey   Ambrose        D                 65625.00      65625.0   
2  ATL      Yamil      Asad        M                150000.00     150000.0   
3  ATL       Mark     Bloom        D                106573.89      99225.0   
4  ATL     Andrew  Carleton        F                 77400.00      65000.0   

                year  
0  mls-salaries-2017  
1  mls-salaries-2017  
2  mls-salaries-2017  
3  mls-salaries-2017  
4  mls-salaries-2017  
(5509, 7)
Rows before dropna(): 5553
Rows after dropna(): 5509
Rows dropped: 44


In [ ]:
# Input features
X = df[[        #array of inputs
    "club",
    "first_name",
    "last_name",
    "position",
    "guaranteed_compensation"
]]

# Output (target)
y = df["base_salary"]

print(X.head())
print(y.head())

  club first_name last_name position  guaranteed_compensation
0  ATL     Miguel   Almiron        M               2297000.00
1  ATL      Mikey   Ambrose        D                 65625.00
2  ATL      Yamil      Asad        M                150000.00
3  ATL       Mark     Bloom        D                106573.89
4  ATL     Andrew  Carleton        F                 77400.00
0    1912500.0
1      65625.0
2     150000.0
3      99225.0
4      65000.0
Name: base_salary, dtype: float64


In [ ]:
from sklearn.model_selection import train_test_split    #splits dataset into training and testing cases

# Split the data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2      # 20% of the data for testing
)

In [ ]:
print("First training example (X_train):")
print(X_train.iloc[0])

print("\nCorresponding target value (y_train):")
print(y_train.iloc[0])

First training example (X_train):
club                            PHI
first_name                  Fabinho
last_name                   Fabinho
position                          D
guaranteed_compensation    118500.0
Name: 5135, dtype: object

Corresponding target value (y_train):
114000.0


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

categorical_features = [
    "club",
    "first_name",
    "last_name",
    "position"
]

numeric_features = [
    "guaranteed_compensation"
]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(predictions[:10])

from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Mean Absolute Error:", mae)
print("R² Score:", r2)


[ 43782.90574797  91906.60068795 186289.70544063 151084.22972325
 898202.95244255 139112.59440363 157236.32009561  78814.57105638
 191433.07468893  33274.47030092]
Mean Absolute Error: 19942.13070873947
R² Score: 0.9854580043251914


In [ ]:
for i in range(len(X_test)):
    first = X_test.iloc[i]["first_name"]
    last = X_test.iloc[i]["last_name"]

    actual = y_test.iloc[i]
    predicted = predictions[i]

    percent_error = abs((actual - predicted) / actual) * 100

    print(f"{first} {last} | Actual: ${actual:,.2f} | Predicted: ${predicted:,.2f} | Error: {percent_error:.2f}%")

Joe Willis | Actual: $46,500.00 | Predicted: $43,782.91 | Error: 5.84%
Robbie Russell | Actual: $95,016.00 | Predicted: $91,906.60 | Error: 3.27%
John Thorrington | Actual: $194,700.00 | Predicted: $186,289.71 | Error: 4.32%
Robbie Rogers | Actual: $160,000.00 | Predicted: $151,084.23 | Error: 5.57%
Lucas Melano | Actual: $790,000.00 | Predicted: $898,202.95 | Error: 13.70%
Richmond Laryea | Actual: $125,000.00 | Predicted: $139,112.59 | Error: 11.29%
Aaron Maund | Actual: $165,000.00 | Predicted: $157,236.32 | Error: 4.71%
Nick Soolsma | Actual: $86,004.00 | Predicted: $78,814.57 | Error: 8.36%
Jermaine Taylor | Actual: $203,500.00 | Predicted: $191,433.07 | Error: 5.93%
Mkhokheli Dube | Actual: $34,650.00 | Predicted: $33,274.47 | Error: 3.97%
Spencer Richey | Actual: $65,004.00 | Predicted: $60,192.03 | Error: 7.40%
Will Bruin | Actual: $120,000.00 | Predicted: $148,867.26 | Error: 24.06%
Chris Tierney | Actual: $34,000.00 | Predicted: $32,698.06 | Error: 3.83%
Michael Boxall | Actu

In [ ]:
#decision tree
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", DecisionTreeRegressor(
        random_state=42,
        max_depth=10   # prevents overfitting (important!)
    ))
])

tree_model.fit(X_train, y_train)
tree_predictions = tree_model.predict(X_test)

mae = mean_absolute_error(y_test, tree_predictions)
r2 = r2_score(y_test, tree_predictions)

print("Decision Tree MAE:", mae)
print("Decision Tree R²:", r2)

for i in range(len(X_test)):
    name = X_test.iloc[i]["first_name"] + " " + X_test.iloc[i]["last_name"]

    actual = y_test.iloc[i]
    predicted = tree_predictions[i]

    print(f"{name} | Actual: ${actual:,.2f} | Predicted: ${predicted:,.2f}")

Decision Tree MAE: 13589.193418315313
Decision Tree R²: 0.9949166373562811
Joe Willis | Actual: $46,500.00 | Predicted: $46,449.39
Robbie Russell | Actual: $95,016.00 | Predicted: $94,038.80
John Thorrington | Actual: $194,700.00 | Predicted: $187,400.74
Robbie Rogers | Actual: $160,000.00 | Predicted: $155,893.50
Lucas Melano | Actual: $790,000.00 | Predicted: $1,000,000.00
Richmond Laryea | Actual: $125,000.00 | Predicted: $138,751.24
Aaron Maund | Actual: $165,000.00 | Predicted: $155,893.50
Nick Soolsma | Actual: $86,004.00 | Predicted: $74,952.40
Jermaine Taylor | Actual: $203,500.00 | Predicted: $187,400.74
Mkhokheli Dube | Actual: $34,650.00 | Predicted: $34,007.61
Spencer Richey | Actual: $65,004.00 | Predicted: $62,269.15
Will Bruin | Actual: $120,000.00 | Predicted: $155,893.50
Chris Tierney | Actual: $34,000.00 | Predicted: $34,007.61
Michael Boxall | Actual: $42,000.00 | Predicted: $41,946.93
Ashtone Morgan | Actual: $100,000.00 | Predicted: $94,038.80
Lawrence Olum | Actua

In [ ]:
from sklearn.metrics import mean_squared_error

# predictions already computed from linear model
train_pred_lr = model.predict(X_train)
test_pred_lr = model.predict(X_test)

mse_train_lr = mean_squared_error(y_train, train_pred_lr)
mse_test_lr = mean_squared_error(y_test, test_pred_lr)

print("Linear Regression MSE (Train):", mse_train_lr)
print("Linear Regression MSE (Test):", mse_test_lr)

train_pred_tree = tree_model.predict(X_train)
test_pred_tree = tree_model.predict(X_test)

mse_train_tree = mean_squared_error(y_train, train_pred_tree)
mse_test_tree = mean_squared_error(y_test, test_pred_tree)

print("Decision Tree MSE (Train):", mse_train_tree)
print("Decision Tree MSE (Test):", mse_test_tree)

Linear Regression MSE (Train): 3124694002.6783214
Linear Regression MSE (Test): 5572174963.0336275
Decision Tree MSE (Train): 171443062.22912794
Decision Tree MSE (Test): 1947833480.6837506


In [ ]:
#Random Forest model
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,   # number of trees
        random_state=42,
        max_depth=None
    ))
])

rf_model.fit(X_train, y_train)
train_pred_rf = rf_model.predict(X_train)
test_pred_rf = rf_model.predict(X_test)

mse_train_rf = mean_squared_error(y_train, train_pred_rf)
mse_test_rf = mean_squared_error(y_test, test_pred_rf)

print("Random Forest MSE (Train):", mse_train_rf)
print("Random Forest MSE (Test):", mse_test_rf)

rf_percent_error = np.abs((y_test - test_pred_rf) / y_test) * 100

print("Average Percent Error (RF):", np.mean(rf_percent_error))

for i in range(len(X_test)):
    name = X_test.iloc[i]["first_name"] + " " + X_test.iloc[i]["last_name"]

    actual = y_test.iloc[i]
    predicted = test_pred_rf[i]

    error = abs((actual - predicted) / actual) * 100

    print(f"{name} | Actual: ${actual:,.2f} | Predicted: ${predicted:,.2f} | Error: {error:.2f}%")

Random Forest MSE (Train): 336692077.89361864
Random Forest MSE (Test): 2287354559.521474
Average Percent Error (RF): 7.1787219258083645
Joe Willis | Actual: $46,500.00 | Predicted: $46,410.04 | Error: 0.19%
Robbie Russell | Actual: $95,016.00 | Predicted: $96,649.37 | Error: 1.72%
John Thorrington | Actual: $194,700.00 | Predicted: $194,622.32 | Error: 0.04%
Robbie Rogers | Actual: $160,000.00 | Predicted: $157,961.16 | Error: 1.27%
Lucas Melano | Actual: $790,000.00 | Predicted: $963,789.99 | Error: 22.00%
Richmond Laryea | Actual: $125,000.00 | Predicted: $140,203.79 | Error: 12.16%
Aaron Maund | Actual: $165,000.00 | Predicted: $162,719.84 | Error: 1.38%
Nick Soolsma | Actual: $86,004.00 | Predicted: $79,105.21 | Error: 8.02%
Jermaine Taylor | Actual: $203,500.00 | Predicted: $202,285.13 | Error: 0.60%
Mkhokheli Dube | Actual: $34,650.00 | Predicted: $34,651.57 | Error: 0.00%
Spencer Richey | Actual: $65,004.00 | Predicted: $64,978.98 | Error: 0.04%
Will Bruin | Actual: $120,000.00

In [ ]:
#KNN model
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import numpy as np

categorical_features = ["club", "first_name", "last_name", "position"]
numeric_features = ["guaranteed_compensation"]

preprocessor_knn = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

knn_model = Pipeline([
    ("preprocessor", preprocessor_knn),
    ("regressor", KNeighborsRegressor(
        n_neighbors=5
    ))
])

knn_model.fit(X_train, y_train)
train_pred_knn = knn_model.predict(X_train)
test_pred_knn = knn_model.predict(X_test)
mse_train_knn = mean_squared_error(y_train, train_pred_knn)
mse_test_knn = mean_squared_error(y_test, test_pred_knn)

print("KNN MSE (Train):", mse_train_knn)
print("KNN MSE (Test):", mse_test_knn)

knn_percent_error = np.abs((y_test - test_pred_knn) / y_test) * 100

print("Average Percent Error (KNN):", np.mean(knn_percent_error))

for i in range(len(X_test)):
    name = X_test.iloc[i]["first_name"] + " " + X_test.iloc[i]["last_name"]

    actual = y_test.iloc[i]
    predicted = test_pred_knn[i]

    error = abs((actual - predicted) / actual) * 100

    print(f"{name} | Actual: ${actual:,.2f} | Predicted: ${predicted:,.2f} | Error: {error:.2f}%")

KNN MSE (Train): 5267529911.627648
KNN MSE (Test): 7574599088.083101
Average Percent Error (KNN): 36.595011003957254
Joe Willis | Actual: $46,500.00 | Predicted: $75,382.50 | Error: 62.11%
Robbie Russell | Actual: $95,016.00 | Predicted: $105,966.69 | Error: 11.53%
John Thorrington | Actual: $194,700.00 | Predicted: $128,600.00 | Error: 33.95%
Robbie Rogers | Actual: $160,000.00 | Predicted: $155,990.00 | Error: 2.51%
Lucas Melano | Actual: $790,000.00 | Predicted: $776,460.00 | Error: 1.71%
Richmond Laryea | Actual: $125,000.00 | Predicted: $130,500.00 | Error: 4.40%
Aaron Maund | Actual: $165,000.00 | Predicted: $52,125.00 | Error: 68.41%
Nick Soolsma | Actual: $86,004.00 | Predicted: $100,000.80 | Error: 16.27%
Jermaine Taylor | Actual: $203,500.00 | Predicted: $135,500.80 | Error: 33.41%
Mkhokheli Dube | Actual: $34,650.00 | Predicted: $51,316.20 | Error: 48.10%
Spencer Richey | Actual: $65,004.00 | Predicted: $64,700.00 | Error: 0.47%
Will Bruin | Actual: $120,000.00 | Predicted: 

In [ ]:
#neural network
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
import numpy as np

categorical_features = ["club", "first_name", "last_name", "position"]
numeric_features = ["guaranteed_compensation"]

preprocessor_nn = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features)
    ]
)

nn_model = Pipeline([
    ("preprocessor", preprocessor_nn),
    ("regressor", MLPRegressor(
        hidden_layer_sizes=(100, 50),  # 2-layer network
        activation="relu",
        max_iter=500,
        random_state=42
    ))
])

nn_model.fit(X_train, y_train)

train_pred_nn = nn_model.predict(X_train)
test_pred_nn = nn_model.predict(X_test)

mse_train_nn = mean_squared_error(y_train, train_pred_nn)
mse_test_nn = mean_squared_error(y_test, test_pred_nn)

print("Neural Network MSE (Train):", mse_train_nn)
print("Neural Network MSE (Test):", mse_test_nn)

nn_percent_error = np.abs((y_test - test_pred_nn) / y_test) * 100

print("Average Percent Error (NN):", np.mean(nn_percent_error))

for i in range(len(X_test)):
    name = X_test.iloc[i]["first_name"] + " " + X_test.iloc[i]["last_name"]

    actual = y_test.iloc[i]
    predicted = test_pred_nn[i]

    error = abs((actual - predicted) / actual) * 100

    print(f"{name} | Actual: ${actual:,.2f} | Predicted: ${predicted:,.2f} | Error: {error:.2f}%")

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Neural Network MSE (Train): 2102191531.726009
Neural Network MSE (Test): 5216918055.761044
Average Percent Error (NN): 29.549112197987718
Joe Willis | Actual: $46,500.00 | Predicted: $57,431.35 | Error: 23.51%
Robbie Russell | Actual: $95,016.00 | Predicted: $131,054.75 | Error: 37.93%
John Thorrington | Actual: $194,700.00 | Predicted: $171,247.68 | Error: 12.05%
Robbie Rogers | Actual: $160,000.00 | Predicted: $152,335.92 | Error: 4.79%
Lucas Melano | Actual: $790,000.00 | Predicted: $956,890.97 | Error: 21.13%
Richmond Laryea | Actual: $125,000.00 | Predicted: $125,249.19 | Error: 0.20%
Aaron Maund | Actual: $165,000.00 | Predicted: $70,853.94 | Error: 57.06%
Nick Soolsma | Actual: $86,004.00 | Predicted: $68,115.48 | Error: 20.80%
Jermaine Taylor | Actual: $203,500.00 | Predicted: $183,464.46 | Error: 9.85%
Mkhokheli Dube | Actual: $34,650.00 | Predicted: $38,090.86 | Error: 9.93%
Spencer Richey | Actual: $65,004.00 | Predicted: $31,076.72 | Error: 52.19%
Will Bruin | Actual: $120,

In [ ]:
#averaging the predictions of all the models
import numpy as np

ensemble_predictions = (
    predictions +
    tree_predictions +
    test_pred_rf +
    test_pred_knn +
    test_pred_nn
) / 5

from sklearn.metrics import mean_squared_error

ensemble_mse = mean_squared_error(y_test, ensemble_predictions)

print("Ensemble Model MSE:", ensemble_mse)

ensemble_train_predictions = (
    train_pred_lr +
    train_pred_tree +
    train_pred_rf +
    train_pred_knn +
    train_pred_nn
) / 5

ensemble_train_mse = mean_squared_error(y_train, ensemble_train_predictions)
ensemble_test_mse = mean_squared_error(y_test, ensemble_predictions)

print("Ensemble Train MSE:", ensemble_train_mse)
print("Ensemble Test MSE:", ensemble_test_mse)

import numpy as np

ensemble_percent_error = np.abs((y_test - ensemble_predictions) / y_test) * 100

print(ensemble_percent_error.head(10))

avg_ensemble_percent_error = np.mean(ensemble_percent_error)

print(f"Ensemble Average Percent Error: {avg_ensemble_percent_error:.2f}%")

Ensemble Model MSE: 2654293999.1844783
Ensemble Train MSE: 1143886520.3294332
Ensemble Test MSE: 2654293999.1844783
4960    15.895136
4317     9.374466
3942    10.820704
1065     3.341899
463     16.337821
1901     7.810691
486     27.414715
3918     6.751206
5488    11.539636
2209    10.441972
Name: base_salary, dtype: float64
Ensemble Average Percent Error: 15.83%


In [ ]:
#to analyze data: ex. could see the average value in each column of datapoints
print("Numeric column averages:")
print(df.mean(numeric_only=True))

print("\nMost common value in each categorical column:")
print(df[["club", "first_name", "last_name", "position"]].mode().iloc[0])
#non-numerical datapoints tough, you could assign a number to each value but not easy because different words have diff meanings
#important to find the distribution of the output values

Numeric column averages:
guaranteed_compensation    205210.729348
base_salary                184826.165211
dtype: float64

Most common value in each categorical column:
club              DAL
first_name      Chris
last_name     Johnson
position            M
Name: 0, dtype: object


In [ ]:
import pandas as pd

# Manual input for Kevin De Bruyne
kevin = pd.DataFrame({
    "club": ["Manchester City"],
    "first_name": ["Kevin"],
    "last_name": ["De Bruyne"],
    "position": ["M"],
    "guaranteed_compensation": [20800000]  # Approximate annual compensation (USD)
})

# Individual model predictions
lr_prediction = model.predict(kevin)[0]
dt_prediction = tree_model.predict(kevin)[0]
rf_prediction = rf_model.predict(kevin)[0]
knn_prediction = knn_model.predict(kevin)[0]
nn_prediction = nn_model.predict(kevin)[0]

# Combined (ensemble) prediction
ensemble_prediction = (
    lr_prediction +
    dt_prediction +
    rf_prediction +
    knn_prediction +
    nn_prediction
) / 5

# Display results
print("Salary Predictions for Kevin De Bruyne")
print("-" * 45)
print(f"Linear Regression : ${lr_prediction:,.2f}")
print(f"Decision Tree     : ${dt_prediction:,.2f}")
print(f"Random Forest     : ${rf_prediction:,.2f}")
print(f"KNN               : ${knn_prediction:,.2f}")
print(f"Neural Network    : ${nn_prediction:,.2f}")
print("-" * 45)
print(f"Ensemble Average  : ${ensemble_prediction:,.2f}")

Salary Predictions for Kevin De Bruyne
---------------------------------------------
Linear Regression : $18,447,733.54
Decision Tree     : $5,600,000.00
Random Forest     : $5,714,115.13
KNN               : $5,422,600.83
Neural Network    : $17,049,856.74
---------------------------------------------
Ensemble Average  : $10,446,861.25


In [ ]:
# Remove guaranteed_compensation from the inputs
X_no_gc = X.drop(columns=["guaranteed_compensation"])

# Create new train/test split
from sklearn.model_selection import train_test_split

X_train_no_gc, X_test_no_gc, y_train_no_gc, y_test_no_gc = train_test_split(
    X_no_gc,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "club",
    "first_name",
    "last_name",
    "position"
]

preprocessor_no_gc = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

model_no_gc = Pipeline([
    ("preprocessor", preprocessor_no_gc),
    ("regressor", LinearRegression())
])

model_no_gc.fit(X_train_no_gc, y_train_no_gc)

train_pred_lr_no_gc = model_no_gc.predict(X_train_no_gc)
test_pred_lr_no_gc = model_no_gc.predict(X_test_no_gc)

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_model_no_gc = Pipeline([
    ("preprocessor", preprocessor_no_gc),
    ("regressor", DecisionTreeRegressor(
        random_state=42,
        max_depth=10
    ))
])

tree_model_no_gc.fit(X_train_no_gc, y_train_no_gc)

train_pred_tree_no_gc = tree_model_no_gc.predict(X_train_no_gc)
test_pred_tree_no_gc = tree_model_no_gc.predict(X_test_no_gc)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model_no_gc = Pipeline([
    ("preprocessor", preprocessor_no_gc),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

rf_model_no_gc.fit(X_train_no_gc, y_train_no_gc)

train_pred_rf_no_gc = rf_model_no_gc.predict(X_train_no_gc)
test_pred_rf_no_gc = rf_model_no_gc.predict(X_test_no_gc)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

knn_model_no_gc = Pipeline([
    ("preprocessor", preprocessor_no_gc),
    ("regressor", KNeighborsRegressor(n_neighbors=5))
])

knn_model_no_gc.fit(X_train_no_gc, y_train_no_gc)

train_pred_knn_no_gc = knn_model_no_gc.predict(X_train_no_gc)
test_pred_knn_no_gc = knn_model_no_gc.predict(X_test_no_gc)

In [ ]:
from sklearn.neural_network import MLPRegressor

nn_model_no_gc = Pipeline([
    ("preprocessor", preprocessor_no_gc),
    ("regressor", MLPRegressor(
        hidden_layer_sizes=(100, 50),
        activation="relu",
        max_iter=500,
        random_state=42
    ))
])

nn_model_no_gc.fit(X_train_no_gc, y_train_no_gc)

train_pred_nn_no_gc = nn_model_no_gc.predict(X_train_no_gc)
test_pred_nn_no_gc = nn_model_no_gc.predict(X_test_no_gc)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
from sklearn.metrics import mean_squared_error

comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "KNN",
        "Neural Network"
    ],
    "Original Test MSE": [
        mse_test_lr,
        mse_test_tree,
        mse_test_rf,
        mse_test_knn,
        mse_test_nn
    ],
    "Without Guaranteed Compensation": [
        mean_squared_error(y_test_no_gc, test_pred_lr_no_gc),
        mean_squared_error(y_test_no_gc, test_pred_tree_no_gc),
        mean_squared_error(y_test_no_gc, test_pred_rf_no_gc),
        mean_squared_error(y_test_no_gc, test_pred_knn_no_gc),
        mean_squared_error(y_test_no_gc, test_pred_nn_no_gc)
    ]
})

print(comparison)

               Model  Original Test MSE  Without Guaranteed Compensation
0  Linear Regression       5.572175e+09                     1.440859e+11
1      Decision Tree       1.947833e+09                     1.982121e+11
2      Random Forest       2.287355e+09                     1.042291e+11
3                KNN       7.574599e+09                     1.873592e+11
4     Neural Network       5.216918e+09                     1.941691e+11


In [ ]:
import numpy as np

# ----- Original ensemble -----
ensemble_predictions = (
    predictions +
    tree_predictions +
    test_pred_rf +
    test_pred_knn +
    test_pred_nn
) / 5

ensemble_percent_error = np.abs((y_test - ensemble_predictions) / y_test) * 100
avg_original_percent_error = np.mean(ensemble_percent_error)

# ----- Ensemble without guaranteed compensation -----
ensemble_predictions_no_gc = (
    test_pred_lr_no_gc +
    test_pred_tree_no_gc +
    test_pred_rf_no_gc +
    test_pred_knn_no_gc +
    test_pred_nn_no_gc
) / 5

ensemble_percent_error_no_gc = np.abs(
    (y_test_no_gc - ensemble_predictions_no_gc) / y_test_no_gc
) * 100

avg_no_gc_percent_error = np.mean(ensemble_percent_error_no_gc)

# ----- Print comparison -----
print("Ensemble Average Percent Error")
print("---------------------------------------")
print(f"Original Model:                 {avg_original_percent_error:.2f}%")
print(f"Without Guaranteed Compensation: {avg_no_gc_percent_error:.2f}%")
print(f"Difference:                     {avg_no_gc_percent_error - avg_original_percent_error:.2f}%")


Ensemble Average Percent Error
---------------------------------------
Original Model:                 15.83%
Without Guaranteed Compensation: 122.50%
Difference:                     106.67%
